In [2]:
! pip install --upgrade astrapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 14.5 MB/s eta 0:00:00


In [ ]:
from astrapy import DataAPIClient

# Initialize the client
client = DataAPIClient("YOUR_TOKEN")
db = client.get_database_by_api_endpoint(
  "https://4c8bfb8e-9fdf-43e7-867d-a7ae457b2703-us-east1.apps.astra.datastax.com",
    keyspace="big_data_fruit",
)

print(f"Connected to Astra DB: {db.list_collection_names()}")

Connected to Astra DB: []


In [21]:
import pandas as pd
df = pd.read_csv("sales_100.csv")

In [22]:
sales_collection = db.create_collection("sales_records")

# Convert to list of dictionaries
data = df.to_dict('records')

# Insert data
result = sales_collection.insert_many(data)
print(f"Inserted {len(result.inserted_ids)} records")

# Query data
print("\nSample records:")
for doc in sales_collection.find().limit(3):
    print(doc)

Inserted 99 records

Sample records:
{'_id': 'a052ca8c-6c79-49bc-92ca-8c6c7989bc08', 'Region': 'Middle East and North Africa', 'Country': 'Algeria', 'Item Type': 'Cosmetics', 'Sales Channel': 'Online', 'Order Priority': 'M', 'Order Date': '2/18/2011', 'Order ID': 761723172, 'Ship Date': '2/24/2011', 'UnitsSold': 9669, 'UnitPrice': 437.2, 'UnitCost': 263.33, 'TotalRevenue': 4227286.8, 'TotalCost': 2546137.77, 'TotalProfit': 1681149.03}
{'_id': '4c667847-c049-4659-a678-47c0491659c3', 'Region': 'Asia', 'Country': 'Kazakhstan', 'Item Type': 'Snacks', 'Sales Channel': 'Online', 'Order Priority': 'L', 'Order Date': '9/8/2013', 'Order ID': 710296428, 'Ship Date': '10/25/2013', 'UnitsSold': 1352, 'UnitPrice': 152.58, 'UnitCost': 97.44, 'TotalRevenue': 206288.16, 'TotalCost': 131738.88, 'TotalProfit': 74549.28}
{'_id': '09b1dede-7490-4384-b1de-de7490a38449', 'Region': 'Europe', 'Country': 'Iceland', 'Item Type': 'Baby Food', 'Sales Channel': 'Offline', 'Order Priority': 'M', 'Order Date': '10/2

In [23]:
# Bronze Layer
bronze = db.create_collection("bronze_sales")
bronze.insert_many(data)

CollectionInsertManyResult(inserted_ids=[2be58f9f-99cd-4291-a58f-9f99cdb29165, a8ab9300-0c3b-4288-ab93-000c3bc28807, ccab468d-fd67-43aa-ab46-8dfd67f3aa28, dd503e9a-e99d-419a-903e-9ae99d819a3b, d59d4ffa-3aa4-4d8a-9d4f-fa3aa44d8a6a ... (99 total)], raw_results=...)

In [25]:
# Create Silver collection if not exists
if "silver_sales" not in db.list_collection_names():
    silver = db.create_collection("silver_sales")
else:
    silver = db.get_collection("silver_sales")

# Get all documents from Bronze
bronze_docs = bronze.find()

for doc in bronze_docs:
    try:
        # Calculate delivery days
        ship_date = pd.to_datetime(doc["Ship Date"])
        order_date = pd.to_datetime(doc["Order Date"])
        delivery_days = (ship_date - order_date).days

        # Calculate profit ratio
        total_revenue = float(doc["TotalRevenue"])
        profit_ratio = round(float(doc["TotalProfit"]) / total_revenue, 4) if total_revenue > 0 else 0.0

        transformed_doc = {
            "order_id": int(doc["Order ID"]),
            "region": doc["Region"].strip(),
            "country": doc["Country"].strip(),
            "item_type": doc["Item Type"].strip(),
            "sales_channel": doc["Sales Channel"].strip(),
            "order_priority": doc["Order Priority"].strip(),
            "order_date": order_date.isoformat(),
            "ship_date": ship_date.isoformat(),
            "units_sold": int(doc["UnitsSold"]),
            "unit_price": float(doc["UnitPrice"]),
            "unit_cost": float(doc["UnitCost"]),
            "profit_ratio": profit_ratio,
            "delivery_days": delivery_days,
            "profit_per_unit": round(float(doc["UnitPrice"]) - float(doc["UnitCost"]), 2)
        }

        silver.insert_one(transformed_doc)

    except Exception as e:
        print(f"Error processing order {doc.get('Order ID')}: {str(e)}")

# count documents in Astra DB
doc_count = len(list(silver.find(limit=10000)))
print(f"Silver layer created with {doc_count} records")

Silver layer created with 99 records


In [ ]:
# verification of silver layer

# Verify first 3 records
print("Sample Silver Records:")
for doc in silver.find(limit=3):
    print(doc)

# Check schema
sample = silver.find_one()
print("\nSchema Example:")
for k, v in sample.items():
    print(f"{k:15} {type(v).__name__:10} {str(v)[:50]}")

Sample Silver Records:
{'_id': '0bcf3f69-aa92-440f-8f3f-69aa92b40f98', 'order_id': 294530856, 'region': 'Europe', 'country': 'Italy', 'item_type': 'Cereal', 'sales_channel': 'Online', 'order_priority': 'M', 'order_date': '2011-11-15T00:00:00', 'ship_date': '2011-12-28T00:00:00', 'units_sold': 7080, 'unit_price': 205.7, 'unit_cost': 117.11, 'profit_ratio': 0.4307, 'delivery_days': 43, 'profit_per_unit': 88.59}
{'_id': '0de606ab-02e9-4f3a-a606-ab02e9df3ae9', 'order_id': 358570849, 'region': 'Middle East and North Africa', 'country': 'Oman', 'item_type': 'Cosmetics', 'sales_channel': 'Online', 'order_priority': 'H', 'order_date': '2010-11-29T00:00:00', 'ship_date': '2010-12-28T00:00:00', 'units_sold': 7937, 'unit_price': 437.2, 'unit_cost': 263.33, 'profit_ratio': 0.3977, 'delivery_days': 29, 'profit_per_unit': 173.87}
{'_id': '244fcace-25ad-498d-8fca-ce25ad998dd9', 'order_id': 839094388, 'region': 'Australia and Oceania', 'country': 'Tonga', 'item_type': 'Baby Food', 'sales_channel': 'On

In [14]:
# Get first document
sample_doc = silver.find_one({})

if sample_doc:
    print("Silver Layer Sample Document:")
    print(sample_doc)

    # Check type
    print("\n Field Type Verification:")
    print(f"unit_price is float: {isinstance(sample_doc.get('unit_price', None), float)}")
    print(f"units_sold is int: {isinstance(sample_doc.get('units_sold', None), int)}")
    print(f"order_date is string: {isinstance(sample_doc.get('order_date', None), str)}")

    # Check date format
    if 'order_date' in sample_doc:
        print(f"order_date contains timestamp: {'T00:00:00' in sample_doc['order_date']}")
    else:
        print("order_date field missing!")
else:
    print("No documents found in silver_sales collection")

Silver Layer Sample Document:
{'_id': '0bcf3f69-aa92-440f-8f3f-69aa92b40f98', 'order_id': 294530856, 'region': 'Europe', 'country': 'Italy', 'item_type': 'Cereal', 'sales_channel': 'Online', 'order_priority': 'M', 'order_date': '2011-11-15T00:00:00', 'ship_date': '2011-12-28T00:00:00', 'units_sold': 7080, 'unit_price': 205.7, 'unit_cost': 117.11, 'profit_ratio': 0.4307, 'delivery_days': 43, 'profit_per_unit': 88.59}

 Field Type Verification:
unit_price is float: True
units_sold is int: True
order_date is string: True
order_date contains timestamp: True


In [26]:
# Get silver sales collection
silver = db.get_collection("silver_sales")

In [29]:
# Gold layer 1: Profit analysis by item_type and region

if "gold_profit_by_region_and_item_type" in db.list_collection_names():
    db.gold_profit_by_region_and_item_type.delete_many({})
else:
    db.create_collection("gold_profit_by_region_and_item_type")

regional_profits = db.get_collection("gold_profit_by_region_and_item_type")

# Aggregation
regions = {}
for doc in silver.find():
    key = (doc["region"], doc["item_type"])
    profit = doc["unit_price"] * doc["units_sold"]

    if key not in regions:
        regions[key] = {
            "total_profit": 0.0,
            "total_units": 0,
            "profit_ratios": []
        }

    regions[key]["total_profit"] += profit
    regions[key]["total_units"] += doc["units_sold"]
    regions[key]["profit_ratios"].append(doc["profit_ratio"])

# Insert results
for (region, item_type), stats in regions.items():
    regional_profits.insert_one({
        "region": region,
        "item_type": item_type,
        "total_profit": round(stats["total_profit"], 2),
        "avg_profit_ratio": round(sum(stats["profit_ratios"])/len(stats["profit_ratios"]), 4),
        "total_units_sold": stats["total_units"]
    })

print(f"Gold layer 1: Profit analysis by item_type and region: {len(regions)} records")


Gold layer 1: Profit analysis by item_type and region: 49 records


In [30]:
# Gold layer 2: Channel Performance

if "gold_channel_performance" in db.list_collection_names():
    db.gold_channel_performance.delete_many({})
else:
    db.create_collection("gold_channel_performance")

channel_performance = db.create_collection("gold_channel_performance")

channels = {}
for doc in silver.find():
    channel = doc["sales_channel"]
    revenue = doc["unit_price"] * doc["units_sold"]

    if channel not in channels:
        channels[channel] = {
            "delivery_days": [],
            "total_revenue": 0.0,
            "order_count": 0
        }

    channels[channel]["delivery_days"].append(doc["delivery_days"])
    channels[channel]["total_revenue"] += revenue
    channels[channel]["order_count"] += 1

# Insert results
for channel, stats in channels.items():
    channel_performance.insert_one({
        "sales_channel": channel,
        "avg_delivery_days": round(sum(stats["delivery_days"])/len(stats["delivery_days"]), 1),
        "total_revenue": round(stats["total_revenue"], 2),
        "order_count": stats["order_count"]
    })

print(f"Gold layer 2: Channel performance: {len(channels)} records")

Gold layer 2: Channel performance: 2 records


In [31]:
# Gold layer 3: Top Products

if "gold_top_products" in db.list_collection_names():
    db.gold_top_products.delete_many({})
else:
    db.create_collection("gold_top_products")

top_products = db.create_collection("gold_top_products")

products = {}
for doc in silver.find():
    product = doc["item_type"]

    if product not in products:
        products[product] = {
            "profits": [],
            "total_units": 0,
            "regions": set()
        }

    products[product]["profits"].append(doc["profit_per_unit"])
    products[product]["total_units"] += doc["units_sold"]
    products[product]["regions"].add(doc["region"])

# Insert results
for product, stats in sorted(
    products.items(),
    key=lambda x: sum(x[1]["profits"])/len(x[1]["profits"]),  # Sort by profitability
    reverse=True
):
    top_products.insert_one({
        "item_type": product,
        "avg_profit_per_unit": round(sum(stats["profits"])/len(stats["profits"]), 2),
        "total_volume": stats["total_units"],
        "region_count": len(stats["regions"])
    })

print(f"Gold layer 3: {len(products)} product rankings")

Gold layer 3: 12 product rankings
